# 05. 벤치마크 & 시각화 — 브로커 성능을 숫자로 비교하기

`난이도: 중급` · `소요: 30분` · `사전 지식:` [01. 프로젝트 개요](./01_project_overview.ipynb) `완료`, 02~04 중 하나 이상 권장

---

## 목차
1. [환경 설정](#1.-환경-설정) (httpx, pandas, matplotlib)
2. [Stopwatch 클래스 분석](#2.-Stopwatch-클래스-분석) — "감"이 아니라 "수치"로 비교하기
3. [5가지 벤치마크 실행](#3.-벤치마크-실행) — Redis Pub/Sub, Redis Stream, RabbitMQ, Kafka, Kafka Batch
4. [pandas DataFrame 변환](#4.-pandas-DataFrame-변환) — 결과를 표로 정리
5. [처리량/지연시간 비교 차트](#5.-matplotlib-시각화) — 막대 차트, P50 vs P99
6. [메시지 수별 스케일링 테스트](#6.-메시지-수별-스케일링-테스트) — 100건 vs 500건 vs 1000건
7. [종합 비교 & 인사이트](#8.-정리-및-핵심-인사이트)

---

### 학습 목표
- "Redis가 빠르다"를 **숫자로 증명**할 수 있게 됩니다
- P50, P99 등 **백분위 지표**의 의미를 이해합니다
- pandas + matplotlib로 벤치마크 결과를 **시각화**합니다

> **참고** — 벤치마크 결과는 실행 환경(Docker 리소스, 네트워크)에 따라 달라집니다.
> 절대값보다 **브로커 간 상대적 비교**에 의미가 있습니다.

---

#### 노트북 네비게이션
| 이전 | 현재 | 다음 |
|------|------|------|
| [04. Kafka Deep Dive](./04_kafka_deep_dive.ipynb) | **05. 벤치마크 & 시각화** | [06. 모니터링 & AOP](./06_monitoring_and_aop.ipynb) |

**전체 학습 경로**: `01 개요` → `02 Redis` · `03 RabbitMQ` · `04 Kafka` → **`05 벤치마크`** → `06 모니터링` → `07 고급패턴` → `08 안정성` → `09 동시성` → `10 Saga`

## 1. 환경 설정

프로젝트 루트를 `sys.path`에 추가하여 `app` 모듈을 임포트할 수 있게 합니다.
벤치마크 API 호출에는 `httpx.AsyncClient`를 사용하고, 결과 분석에는 `pandas`와 `matplotlib`를 사용합니다.

> **참고:** 한글 폰트가 matplotlib에서 깨지는 경우,
> `plt.rcParams['font.family'] = 'AppleGothic'` (macOS) 또는
> `plt.rcParams['font.family'] = 'Malgun Gothic'` (Windows)를 설정하세요.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = str(Path.cwd().parent)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import httpx
import pandas as pd
import matplotlib.pyplot as plt

BASE_URL = "http://localhost:8000"
client = httpx.AsyncClient(base_url=BASE_URL, timeout=60.0)  # 벤치마크는 시간이 오래 걸릴 수 있음

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

### matplotlib 스타일 설정

깔끔한 그래프를 위해 seaborn 스타일을 적용합니다.
색상 팔레트도 미리 정의하여 브로커별로 일관된 색상을 사용합니다.

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')

BROKER_COLORS = {
    'redis_pubsub': '#e74c3c',
    'redis_stream': '#e67e22',
    'rabbitmq': '#2ecc71',
    'kafka': '#3498db',
    'kafka_batch': '#9b59b6',
}

print("환경 설정 완료!")

## 2. Stopwatch 클래스 분석

### 왜 벤치마크를 해야 하나?

"Redis가 빠르다", "Kafka가 대용량에 강하다"는 말을 많이 들어보셨을 것입니다.
하지만 **"얼마나 빠른데?"**, **"우리 환경에서도 그런가?"**라는 질문에는 숫자로 답해야 합니다.

```text
  "Redis가 빠르다" → 얼마나? → 측정해보니 0.3ms/건
  "Kafka가 느리다" → 정말?  → 배치로 보내니 오히려 더 빠름!
  
  벤치마크 = "감"이 아니라 "수치"로 비교하는 것
```

### Stopwatch의 역할

이 프로젝트의 `Stopwatch` 클래스는 **스톱워치**처럼 동작합니다.

```text
  일반 스톱워치:
  [시작] ──→ [랩 1: 2.3초] ──→ [랩 2: 1.8초] ──→ [랩 3: 2.1초] ──→ [정지]
                                                                    ↓
                                                        평균: 2.07초
                                                        최소: 1.8초
                                                        최대: 2.3초

  Stopwatch 클래스:
  sw.start() → with sw.lap(): 메시지발행 → ... → sw.stop()
                                                      ↓
                                             sw.report() = {
                                               avg_latency_ms: 0.5,
                                               min_latency_ms: 0.3,
                                               max_latency_ms: 1.2,
                                               p50_latency_ms: 0.45,
                                               p99_latency_ms: 1.1,
                                               throughput_msg_per_sec: 2000
                                             }
```

### P50, P99이 뭔가요?

평균(avg)만 보면 함정에 빠질 수 있습니다.

```text
  예시: 10번 측정한 결과 (ms)
  [0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 50.0]
  
  평균: 5.3ms ← 이것만 보면 "좀 느리네" 라고 생각
  P50:  0.3ms ← 절반의 요청은 0.3ms 이하 (실제 체감 속도)
  P99:  50ms  ← 100번 중 1번은 50ms까지 걸림 (최악의 경우)
  
  해석: "보통은 0.3ms로 매우 빠르지만,
         가끔 50ms짜리 이상치(outlier)가 있다"
```

| 지표 | 의미 | 비유 |
|------|------|------|
| **P50 (중앙값)** | 절반의 요청이 이 시간 이하 | "보통 이 정도 걸린다" |
| **P99** | 99%의 요청이 이 시간 이하 | "최악의 경우 이 정도 걸린다" |
| **평균** | 전체 합계 / 개수 | 이상치에 왜곡되기 쉬움 |

> **실무 팁** — SLA(서비스 수준 계약)에서는 보통 P99를 기준으로 합니다.
> "P99 < 100ms"는 "100번 중 99번은 100ms 이내에 응답한다"는 의미입니다.

아래에서 `Stopwatch`의 소스코드를 직접 확인해봅시다.

In [ ]:
import inspect
from app.monitoring.timer import Stopwatch

source = inspect.getsource(Stopwatch)
print(source)

### Stopwatch의 `lap()` 동작 원리

`lap()`은 **컨텍스트 매니저**로 구현되어 있습니다.
`with sw.lap():` 블록 안의 코드 실행 시간을 측정하여 `latencies` 리스트에 추가합니다.

간단한 예제로 동작을 확인해 봅시다.

In [ ]:
import time

sw = Stopwatch()
sw.start()

for _ in range(5):
    with sw.lap():
        time.sleep(0.01)  # 10ms 대기 시뮬레이션

sw.stop()
print(f"측정된 latencies: {sw.latencies}")
print(f"총 소요 시간: {sw.total_ms:.2f}ms")

### `report()` 메서드 확인

`report()`는 수집된 latencies로부터 평균, 최소, 최대, p50, p99, 처리량 등 핵심 통계를 산출합니다.
벤치마크 API 응답이 바로 이 형식의 dict를 반환합니다.

In [ ]:
report = sw.report(broker="test", count=5)

for key, value in report.items():
    print(f"  {key}: {value}")

## 3. 벤치마크 실행

이제 FastAPI 서버의 벤치마크 엔드포인트를 호출하여 실제 메시지 브로커 성능을 측정합니다.
먼저 **100개 메시지**로 빠르게 결과를 확인합니다.

> **사전 조건:** FastAPI 서버(`uvicorn app.main:app`)와 모든 브로커(Redis, RabbitMQ, Kafka)가 실행 중이어야 합니다.

### 3-1. Redis Pub/Sub 벤치마크

Redis Pub/Sub은 가장 단순한 메시지 전달 모델입니다.
메시지가 발행되면 연결된 모든 구독자에게 즉시 전달되며, 영속성이 없습니다.

In [ ]:
MESSAGE_COUNT = 100

resp = await client.post(
    "/benchmark/redis",
    json={"message_count": MESSAGE_COUNT}
)
redis_pubsub_result = resp.json()
print(f"Redis Pub/Sub ({MESSAGE_COUNT}건):")
print(f"  처리량: {redis_pubsub_result.get('throughput_msg_per_sec', 'N/A')} msg/sec")
print(f"  평균 지연: {redis_pubsub_result.get('avg_latency_ms', 'N/A')} ms")

### 3-2. Redis Stream 벤치마크

Redis Stream은 Pub/Sub과 달리 메시지를 영속적으로 저장하며, 소비자 그룹을 지원합니다.
저장 오버헤드로 인해 Pub/Sub보다 약간 느릴 수 있습니다.

In [ ]:
resp = await client.post(
    "/benchmark/redis-stream",
    json={"message_count": MESSAGE_COUNT}
)
redis_stream_result = resp.json()
print(f"Redis Stream ({MESSAGE_COUNT}건):")
print(f"  처리량: {redis_stream_result.get('throughput_msg_per_sec', 'N/A')} msg/sec")
print(f"  평균 지연: {redis_stream_result.get('avg_latency_ms', 'N/A')} ms")

### 3-3. RabbitMQ 벤치마크

RabbitMQ는 AMQP 프로토콜 기반의 전통적인 메시지 브로커입니다.
메시지 라우팅, 확인(ack), 큐 기반 전달을 지원합니다.

In [ ]:
resp = await client.post(
    "/benchmark/rabbitmq",
    json={"message_count": MESSAGE_COUNT}
)
rabbitmq_result = resp.json()
print(f"RabbitMQ ({MESSAGE_COUNT}건):")
print(f"  처리량: {rabbitmq_result.get('throughput_msg_per_sec', 'N/A')} msg/sec")
print(f"  평균 지연: {rabbitmq_result.get('avg_latency_ms', 'N/A')} ms")

### 3-4. Kafka 벤치마크

Kafka는 분산 로그 기반 메시지 시스템입니다.
개별 메시지 전송 시 파티션 리더와의 통신 오버헤드가 있을 수 있습니다.

In [ ]:
resp = await client.post(
    "/benchmark/kafka",
    json={"message_count": MESSAGE_COUNT}
)
kafka_result = resp.json()
print(f"Kafka ({MESSAGE_COUNT}건):")
print(f"  처리량: {kafka_result.get('throughput_msg_per_sec', 'N/A')} msg/sec")
print(f"  평균 지연: {kafka_result.get('avg_latency_ms', 'N/A')} ms")

### 3-5. Kafka Batch 벤치마크

Kafka의 배치 전송은 여러 메시지를 한 번에 묶어 전송합니다.
네트워크 라운드트립 횟수가 줄어들어 처리량이 크게 향상될 수 있습니다.

In [ ]:
resp = await client.post(
    "/benchmark/kafka-batch",
    json={"message_count": MESSAGE_COUNT}
)
kafka_batch_result = resp.json()
print(f"Kafka Batch ({MESSAGE_COUNT}건):")
print(f"  처리량: {kafka_batch_result.get('throughput_msg_per_sec', 'N/A')} msg/sec")
print(f"  평균 지연: {kafka_batch_result.get('avg_latency_ms', 'N/A')} ms")

## 4. pandas DataFrame 변환

5가지 벤치마크 결과를 하나의 DataFrame으로 합쳐서 한눈에 비교합니다.
이후 시각화에서도 이 DataFrame을 기반으로 차트를 그립니다.

In [ ]:
all_results = [
    redis_pubsub_result,
    redis_stream_result,
    rabbitmq_result,
    kafka_result,
    kafka_batch_result,
]

df = pd.DataFrame(all_results)
df = df.set_index('broker')
df

### 핵심 지표 요약

처리량과 지연시간 지표만 추출하여 정렬합니다.
처리량이 높을수록, 지연시간이 낮을수록 좋은 성능입니다.

In [ ]:
summary_cols = [
    'throughput_msg_per_sec',
    'avg_latency_ms',
    'p50_latency_ms',
    'p99_latency_ms',
]
df_summary = df[summary_cols].sort_values(
    'throughput_msg_per_sec', ascending=False
)
print("=== 성능 요약 (처리량 내림차순) ===")
df_summary

## 5. matplotlib 시각화

### 5-1. 처리량(Throughput) 막대 차트

각 브로커의 초당 처리 메시지 수를 비교합니다.
처리량이 높을수록 대량 메시지 처리에 유리합니다.

In [ ]:
fig, ax = plt.subplots()
brokers = df.index.tolist()
colors = [BROKER_COLORS.get(b, '#95a5a6') for b in brokers]
throughputs = df['throughput_msg_per_sec']

bars = ax.bar(brokers, throughputs, color=colors, edgecolor='white')
ax.set_ylabel('Messages / sec')
ax.set_title('Broker Throughput Comparison (msg/sec)')

for bar, val in zip(bars, throughputs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
            f'{val:,.0f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

### 5-2. 평균 지연시간 비교 차트

각 브로커의 메시지 1건당 평균 발행 지연시간을 비교합니다.
지연시간이 낮을수록 실시간 처리에 적합합니다.

In [ ]:
fig, ax = plt.subplots()
avg_latencies = df['avg_latency_ms']

bars = ax.barh(brokers, avg_latencies, color=colors, edgecolor='white')
ax.set_xlabel('Average Latency (ms)')
ax.set_title('Broker Average Latency Comparison')

for bar, val in zip(bars, avg_latencies):
    ax.text(bar.get_width(), bar.get_y() + bar.get_height()/2,
            f' {val:.2f}ms', ha='left', va='center', fontsize=10)

plt.tight_layout()
plt.show()

### 5-3. P50 vs P99 지연시간 비교 차트

**P50(중앙값)**과 **P99(99번째 백분위)**를 비교하면 **꼬리 지연(tail latency)**을 파악할 수 있습니다.

```text
  P50과 P99의 차이가 작은 브로커:
  ┃████████████████████┃  P50 = 0.5ms
  ┃█████████████████████┃ P99 = 0.7ms   ← 안정적! 거의 일정한 속도
  
  P50과 P99의 차이가 큰 브로커:
  ┃████┃                  P50 = 0.3ms
  ┃███████████████████████████████┃ P99 = 5ms  ← 불안정! 가끔 매우 느림
```

P99가 P50보다 **10배 이상** 크다면, 간헐적으로 매우 느린 요청이 발생한다는 의미입니다.
이런 브로커는 실시간 서비스에서 사용자 경험을 해칠 수 있습니다.

In [ ]:
import numpy as np

fig, ax = plt.subplots()
x = np.arange(len(brokers))
width = 0.35

bars_p50 = ax.bar(x - width/2, df['p50_latency_ms'], width,
                   label='P50', color='#3498db', alpha=0.8)
bars_p99 = ax.bar(x + width/2, df['p99_latency_ms'], width,
                   label='P99', color='#e74c3c', alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(brokers, rotation=15)
ax.set_ylabel('Latency (ms)')
ax.set_title('P50 vs P99 Latency by Broker')
ax.legend()
plt.tight_layout()
plt.show()

## 6. 메시지 수별 스케일링 테스트

### 스케일링 테스트란?

"100건 처리할 때 빠르면, 10000건도 빠를까?" — 이 질문에 답하는 테스트입니다.

```text
  이상적인 브로커 (선형 스케일링):
  100건  → 1000 msg/sec
  500건  → 1000 msg/sec    ← 처리량이 일정!
  1000건 → 1000 msg/sec
  
  문제 있는 브로커:
  100건  → 1000 msg/sec
  500건  → 800 msg/sec     ← 처리량이 떨어지기 시작
  1000건 → 400 msg/sec     ← 절반으로 감소! (병목 발생)
```

처리량이 메시지 수에 관계없이 **일정하면 좋은 것**이고,
메시지가 많아질수록 **떨어지면 어딘가에 병목**이 있다는 신호입니다.

메시지 수를 **100 → 500 → 1000**으로 증가시키며 관찰합니다.

> **주의:** 이 테스트는 시간이 다소 걸릴 수 있습니다 (모든 브로커 × 3회 = 15회 벤치마크).

### 스케일링 테스트용 헬퍼 함수

반복적인 API 호출을 함수로 추상화하여 코드 중복을 줄입니다.

In [ ]:
async def run_benchmark(endpoint: str, count: int) -> dict:
    """벤치마크 엔드포인트를 호출하고 결과를 반환합니다."""
    resp = await client.post(
        endpoint,
        json={"message_count": count}
    )
    return resp.json()

ENDPOINTS = {
    'redis_pubsub': '/benchmark/redis',
    'redis_stream': '/benchmark/redis-stream',
    'rabbitmq': '/benchmark/rabbitmq',
    'kafka': '/benchmark/kafka',
    'kafka_batch': '/benchmark/kafka-batch',
}

### 스케일링 테스트 실행

3가지 메시지 수(100, 500, 1000)에 대해 모든 브로커를 순차적으로 테스트합니다.
결과를 리스트에 수집하여 나중에 DataFrame으로 변환합니다.

In [ ]:
SCALE_COUNTS = [100, 500, 1000]
scaling_results = []

for count in SCALE_COUNTS:
    print(f"\n--- {count}건 테스트 ---")
    for name, endpoint in ENDPOINTS.items():
        result = await run_benchmark(endpoint, count)
        result['test_count'] = count
        scaling_results.append(result)
        tp = result.get('throughput_msg_per_sec', 'N/A')
        print(f"  {name}: {tp} msg/sec")

print("\n스케일링 테스트 완료!")

### 스케일링 결과를 DataFrame으로 변환

스케일링 테스트 결과를 DataFrame으로 변환하고 피벗 테이블을 만들어
브로커별 메시지 수 변화에 따른 처리량을 정리합니다.

In [ ]:
df_scale = pd.DataFrame(scaling_results)

pivot_throughput = df_scale.pivot_table(
    index='test_count',
    columns='broker',
    values='throughput_msg_per_sec'
)

pivot_latency = df_scale.pivot_table(
    index='test_count',
    columns='broker',
    values='avg_latency_ms'
)

print("=== 메시지 수별 처리량 (msg/sec) ===")
pivot_throughput

## 7. 스케일링 결과 시각화

### 7-1. 처리량 스케일링 라인 차트

메시지 수가 증가할 때 각 브로커의 처리량이 어떻게 변화하는지 라인 차트로 표현합니다.
이상적인 경우 처리량은 메시지 수와 무관하게 일정해야 합니다.

In [ ]:
fig, ax = plt.subplots()

for broker_name in pivot_throughput.columns:
    color = BROKER_COLORS.get(broker_name, '#95a5a6')
    ax.plot(pivot_throughput.index, pivot_throughput[broker_name],
            marker='o', label=broker_name, color=color, linewidth=2)

ax.set_xlabel('Message Count')
ax.set_ylabel('Throughput (msg/sec)')
ax.set_title('Throughput Scaling by Message Count')
ax.legend(loc='best')
ax.set_xticks(SCALE_COUNTS)
plt.tight_layout()
plt.show()

### 7-2. 평균 지연시간 스케일링 라인 차트

메시지 수 증가에 따른 평균 지연시간 변화를 관찰합니다.
메시지 수가 많아져도 지연시간이 크게 증가하지 않으면 안정적인 브로커라 할 수 있습니다.

In [ ]:
fig, ax = plt.subplots()

for broker_name in pivot_latency.columns:
    color = BROKER_COLORS.get(broker_name, '#95a5a6')
    ax.plot(pivot_latency.index, pivot_latency[broker_name],
            marker='s', label=broker_name, color=color, linewidth=2)

ax.set_xlabel('Message Count')
ax.set_ylabel('Average Latency (ms)')
ax.set_title('Average Latency Scaling by Message Count')
ax.legend(loc='best')
ax.set_xticks(SCALE_COUNTS)
plt.tight_layout()
plt.show()

### 7-3. 종합 비교: 처리량 vs 지연시간 산점도

처리량(x축)과 평균 지연시간(y축)을 한 차트에 표시하여
**고처리량 + 저지연** 영역(오른쪽 아래)에 위치한 브로커가 가장 효율적임을 확인합니다.

In [ ]:
fig, ax = plt.subplots()

for _, row in df.iterrows():
    color = BROKER_COLORS.get(row.name, '#95a5a6')
    ax.scatter(row['throughput_msg_per_sec'], row['avg_latency_ms'],
               s=150, color=color, edgecolors='black', zorder=5)
    ax.annotate(row.name, (row['throughput_msg_per_sec'], row['avg_latency_ms']),
                textcoords='offset points', xytext=(8, 5), fontsize=9)

ax.set_xlabel('Throughput (msg/sec)')
ax.set_ylabel('Avg Latency (ms)')
ax.set_title('Throughput vs Latency (lower-right is better)')
plt.tight_layout()
plt.show()

## 8. 정리 및 핵심 인사이트

### 벤치마크 결과 최종 요약

모든 결과를 하나의 테이블로 정리합니다.

In [ ]:
print("="*60)
print("         벤치마크 최종 요약")
print("="*60)

best_throughput = df['throughput_msg_per_sec'].idxmax()
best_latency = df['avg_latency_ms'].idxmin()
best_p99 = df['p99_latency_ms'].idxmin()

print(f"  최고 처리량:       {best_throughput}")
print(f"  최저 평균 지연:    {best_latency}")
print(f"  최저 P99 지연:     {best_p99}")
print("="*60)

### 클라이언트 정리

httpx 비동기 클라이언트를 명시적으로 닫아 리소스를 해제합니다.

In [ ]:
await client.aclose()
print("httpx 클라이언트가 정상적으로 종료되었습니다.")

### 핵심 인사이트

| 관점 | 권장 브로커 | 이유 |
|------|-----------|------|
| **최대 처리량** | Kafka Batch | 배치 전송으로 네트워크 오버헤드 최소화 |
| **최저 지연시간** | Redis Pub/Sub | 인메모리 기반, 프로토콜 오버헤드 최소 |
| **안정성 (낮은 P99)** | Redis Stream | 영속성 + 안정적인 꼬리 지연 |
| **범용 메시징** | RabbitMQ | 라우팅, ACK, 큐 기반 안정적 전달 |
| **대용량 스트리밍** | Kafka | 파티션 기반 수평 확장, 로그 영속성 |

---

**다음 단계:**
- 동시 소비자(consumer) 수를 늘려 소비 성능도 측정해 보세요.
- 메시지 크기(payload size)를 변경하며 테스트하면 더 풍부한 비교가 가능합니다.
- `/monitoring/comparison` 엔드포인트로 서버 측 집계 결과도 확인해 보세요.